In [0]:
dbutils.widgets.text("fecha_proceso", "")
fecha_proceso = dbutils.widgets.get("fecha_proceso")
print("Fecha de proceso:", fecha_proceso)

Fecha de proceso: 


In [0]:
# Listar schemas en tu catálogo
spark.sql("SHOW SCHEMAS IN cat_asolisr_20260622").show()


+------------------+
|      databaseName|
+------------------+
|            bronze|
|           default|
|              gold|
|information_schema|
|            silver|
+------------------+



In [0]:
df = (spark.read
      .option("header", "true")
      .option("inferSchema", "true")
      .csv("/Volumes/cat_asolisr_20260622/default/vol_landing/clientes_logistica.csv"))

df.show()

+----------+--------------------+--------+----------------+----------+
|id_cliente|              nombre|segmento|          ciudad|fecha_alta|
+----------+--------------------+--------+----------------+----------+
|   CLI-001|Exportadora Guada...| Empresa|     Guadalajara|2022-03-26|
|   CLI-002|Proveedora Monter...| Empresa|       Monterrey|2021-04-25|
|   CLI-003| Cliente Persona 003| Persona|          Puebla|2021-03-07|
|   CLI-004|Industrials Verac...| Empresa|        Veracruz|2022-10-09|
|   CLI-005|Distribuidora Tij...| Empresa|         Tijuana|2022-07-03|
|   CLI-006|Exportadora Mérid...| Empresa|          Mérida|2022-03-09|
|   CLI-007| Cliente Persona 007| Persona|Ciudad de México|2022-03-12|
|   CLI-008|Industrials Guada...| Empresa|     Guadalajara|2023-10-20|
|   CLI-009|Comercial Monterr...| Empresa|       Monterrey|2023-07-29|
|   CLI-010|Comercial Puebla #10| Empresa|          Puebla|2022-06-27|
|   CLI-011| Cliente Persona 011| Persona|        Veracruz|2022-05-21|
|   CL

In [0]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import current_timestamp

catalog = "cat_asolisr_20260622"
volume_path = f"/Volumes/{catalog}/default/vol_landing"

# Crear esquema bronze si no existe
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze")

# Esquemas explícitos
schemas = {

    "clientes_logistica": StructType([
        StructField("id_cliente", StringType(), True),
        StructField("nombre", StringType(), True),
        StructField("segmento", StringType(), True),
        StructField("ciudad", StringType(), True),
        StructField("fecha_alta", StringType(), True)
    ]),

    "envios": StructType([
        StructField("id_envio", StringType(), True),
        StructField("id_cliente", StringType(), True),
        StructField("id_transportista", StringType(), True),
        StructField("id_ruta", StringType(), True),
        StructField("peso_kg", StringType(), True),
        StructField("monto_flete", StringType(), True),
        StructField("fecha_envio", StringType(), True),
        StructField("estado", StringType(), True),
        StructField("updated_at", StringType(), True)
    ]),

    "incidencias_2023": StructType([
        StructField("id_incidencia", StringType(), True),
        StructField("id_envio", StringType(), True),
        StructField("tipo_incidencia", StringType(), True),
        StructField("fecha_incidencia", StringType(), True),
        StructField("resolucion", StringType(), True),
        StructField("estado_resolucion", StringType(), True)
    ]),

    "incidencias_2024": StructType([
        StructField("id_incidencia", StringType(), True),
        StructField("id_envio", StringType(), True),
        StructField("tipo_incidencia", StringType(), True),
        StructField("fecha_incidencia", StringType(), True),
        StructField("resolucion", StringType(), True),
        StructField("estado_resolucion", StringType(), True)
    ]),

    "rutas": StructType([
        StructField("id_ruta", StringType(), True),
        StructField("origen", StringType(), True),
        StructField("destino", StringType(), True),
        StructField("tipo_ruta", StringType(), True),
        StructField("distancia_km", StringType(), True)
    ]),

    "transportistas": StructType([
        StructField("id_transportista", StringType(), True),
        StructField("nombre", StringType(), True),
        StructField("zona", StringType(), True),
        StructField("ciudad", StringType(), True),
        StructField("modalidad", StringType(), True)
    ])
}

for table_name, schema in schemas.items():

    try:

        csv_path = f"{volume_path}/{table_name}.csv"
        table_full_name = f"{catalog}.bronze.bronze_{table_name}"

        print(f"Leyendo: {csv_path}")

        df = (
            spark.read
            .option("header", "true")
            .schema(schema)
            .csv(csv_path)
            .withColumn("ingestion_timestamp", current_timestamp())
        )

        # Eliminar tabla previa para evitar conflictos de metadata
        spark.sql(f"DROP TABLE IF EXISTS {table_full_name}")

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(table_full_name)
        )

        print(f"✓ Tabla creada: {table_full_name}")
        print(f"  Registros: {df.count()}")

    except Exception as e:
        print(f"✗ Error en {table_name}")
        print(str(e))


Leyendo: /Volumes/cat_asolisr_20260622/default/vol_landing/clientes_logistica.csv
✓ Tabla creada: cat_asolisr_20260622.bronze.bronze_clientes_logistica
  Registros: 60
Leyendo: /Volumes/cat_asolisr_20260622/default/vol_landing/envios.csv
✓ Tabla creada: cat_asolisr_20260622.bronze.bronze_envios
  Registros: 600
Leyendo: /Volumes/cat_asolisr_20260622/default/vol_landing/incidencias_2023.csv
✓ Tabla creada: cat_asolisr_20260622.bronze.bronze_incidencias_2023
  Registros: 120
Leyendo: /Volumes/cat_asolisr_20260622/default/vol_landing/incidencias_2024.csv
✓ Tabla creada: cat_asolisr_20260622.bronze.bronze_incidencias_2024
  Registros: 80
Leyendo: /Volumes/cat_asolisr_20260622/default/vol_landing/rutas.csv
✓ Tabla creada: cat_asolisr_20260622.bronze.bronze_rutas
  Registros: 30
Leyendo: /Volumes/cat_asolisr_20260622/default/vol_landing/transportistas.csv
✓ Tabla creada: cat_asolisr_20260622.bronze.bronze_transportistas
  Registros: 20


In [0]:
from pyspark.sql import functions as F

catalog = "cat_asolisr_20260622"

# Evita errores de parseo y devuelve NULL cuando una fecha no coincide
spark.conf.set("spark.sql.ansi.enabled", "false")

# Crear esquema Silver
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver")

# =====================================================
# SILVER ENVIOS
# =====================================================

bronze_envios = spark.table(f"{catalog}.bronze.bronze_envios")

silver_envios = (
    bronze_envios

    .withColumn(
        "fecha_envio",
        F.coalesce(
            F.to_date("fecha_envio", "yyyy-MM-dd"),
            F.to_date("fecha_envio", "dd/MM/yyyy"),
            F.to_date("fecha_envio", "MM/dd/yyyy"),
            F.to_date("fecha_envio", "yyyy/MM/dd"),
            F.to_date("fecha_envio", "dd-MM-yyyy")
        )
    )

    .withColumn(
        "peso_kg",
        F.when(
            F.upper(F.trim(F.col("peso_kg"))) == "N/A",
            None
        ).otherwise(
            F.col("peso_kg").cast("double")
        )
    )

    .withColumn(
        "monto_flete",
        F.when(
            F.upper(F.trim(F.col("monto_flete"))) == "N/A",
            None
        ).otherwise(
            F.col("monto_flete").cast("double")
        )
    )

    .withColumn(
        "estado",
        F.lower(F.trim(F.col("estado")))
    )

    .dropDuplicates(["id_envio"])
)

(
    silver_envios.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.silver_envios")
)

print("✓ silver_envios creada")


# =====================================================
# SILVER CLIENTES
# =====================================================

bronze_clientes = spark.table(
    f"{catalog}.bronze.bronze_clientes_logistica"
)

silver_clientes = (
    bronze_clientes

    .withColumn(
        "fecha_alta",
        F.coalesce(
            F.to_date("fecha_alta", "yyyy-MM-dd"),
            F.to_date("fecha_alta", "dd/MM/yyyy"),
            F.to_date("fecha_alta", "MM/dd/yyyy"),
            F.to_date("fecha_alta", "yyyy/MM/dd"),
            F.to_date("fecha_alta", "dd-MM-yyyy")
        )
    )

    .dropDuplicates(["id_cliente"])
)

(
    silver_clientes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalog}.silver.silver_clientes_logistica"
    )
)

print("✓ silver_clientes_logistica creada")


# =====================================================
# SILVER INCIDENCIAS
# SOLO incidencias_2023
# =====================================================

bronze_incidencias = spark.table(
    f"{catalog}.bronze.bronze_incidencias_2023"
)

silver_incidencias = (
    bronze_incidencias

    .withColumn(
        "fecha_incidencia",
        F.coalesce(
            F.to_date("fecha_incidencia", "yyyy-MM-dd"),
            F.to_date("fecha_incidencia", "dd/MM/yyyy"),
            F.to_date("fecha_incidencia", "MM/dd/yyyy"),
            F.to_date("fecha_incidencia", "yyyy/MM/dd"),
            F.to_date("fecha_incidencia", "dd-MM-yyyy")
        )
    )

    .dropDuplicates(["id_incidencia"])
)

(
    silver_incidencias.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalog}.silver.silver_incidencias"
    )
)

print("✓ silver_incidencias creada")


# =====================================================
# SILVER RUTAS
# =====================================================

bronze_rutas = spark.table(
    f"{catalog}.bronze.bronze_rutas"
)

silver_rutas = (
    bronze_rutas

    .withColumn(
        "distancia_km",
        F.col("distancia_km").cast("double")
    )

    .dropDuplicates(["id_ruta"])
)

(
    silver_rutas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalog}.silver.silver_rutas"
    )
)

print("✓ silver_rutas creada")


# =====================================================
# SILVER TRANSPORTISTAS
# =====================================================

bronze_transportistas = spark.table(
    f"{catalog}.bronze.bronze_transportistas"
)

silver_transportistas = (
    bronze_transportistas
    .dropDuplicates(["id_transportista"])
)

(
    silver_transportistas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalog}.silver.silver_transportistas"
    )
)

print("✓ silver_transportistas creada")


# =====================================================
# VALIDACIÓN
# =====================================================

tablas = [
    "silver_envios",
    "silver_clientes_logistica",
    "silver_incidencias",
    "silver_rutas",
    "silver_transportistas"
]

for tabla in tablas:
    print(f"\n===== {tabla} =====")
    spark.table(f"{catalog}.silver.{tabla}").printSchema()
    print(
        f"Registros: {spark.table(f'{catalog}.silver.{tabla}').count()}"
    )

✓ silver_envios creada
✓ silver_clientes_logistica creada
✓ silver_incidencias creada
✓ silver_rutas creada
✓ silver_transportistas creada

===== silver_envios =====
root
 |-- id_envio: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- id_transportista: string (nullable = true)
 |-- id_ruta: string (nullable = true)
 |-- peso_kg: double (nullable = true)
 |-- monto_flete: double (nullable = true)
 |-- fecha_envio: date (nullable = true)
 |-- estado: string (nullable = true)
 |-- updated_at: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)

Registros: 600

===== silver_clientes_logistica =====
root
 |-- id_cliente: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- fecha_alta: date (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)

Registros: 60

===== silver_incidencias =====
root
 |-- id_incidencia: string (nullable = 

In [0]:
from pyspark.sql import functions as F

catalog = "cat_asolisr_20260622"

# =====================================================
# 1. OBTENER VERSIÓN ACTUAL ANTES DEL MERGE
# =====================================================

history_before = spark.sql(f"""
DESCRIBE HISTORY {catalog}.silver.silver_incidencias
""")

version_before = history_before.select("version").first()[0]

print(f"Versión antes del MERGE: {version_before}")

# =====================================================
# 2. LEER INCIDENCIAS 2024 DESDE BRONZE
# =====================================================

spark.conf.set("spark.sql.ansi.enabled", "false")

incidencias_2024 = (
    spark.table(f"{catalog}.bronze.bronze_incidencias_2024")

    .withColumn(
        "fecha_incidencia",
        F.coalesce(
            F.to_date("fecha_incidencia", "yyyy-MM-dd"),
            F.to_date("fecha_incidencia", "dd/MM/yyyy"),
            F.to_date("fecha_incidencia", "MM/dd/yyyy"),
            F.to_date("fecha_incidencia", "yyyy/MM/dd"),
            F.to_date("fecha_incidencia", "dd-MM-yyyy")
        )
    )

    .dropDuplicates(["id_incidencia"])
)

incidencias_2024.createOrReplaceTempView(
    "stg_incidencias_2024"
)

# =====================================================
# 3. MERGE INTO
# =====================================================

spark.sql(f"""
MERGE INTO {catalog}.silver.silver_incidencias AS tgt
USING stg_incidencias_2024 AS src
ON tgt.id_incidencia = src.id_incidencia

WHEN MATCHED THEN
UPDATE SET
    tgt.estado_resolucion = src.estado_resolucion,
    tgt.resolucion = src.resolucion

WHEN NOT MATCHED THEN
INSERT (
    id_incidencia,
    id_envio,
    tipo_incidencia,
    fecha_incidencia,
    resolucion,
    estado_resolucion,
    ingestion_timestamp
)
VALUES (
    src.id_incidencia,
    src.id_envio,
    src.tipo_incidencia,
    src.fecha_incidencia,
    src.resolucion,
    src.estado_resolucion,
    src.ingestion_timestamp
)
""")

print("✓ MERGE ejecutado correctamente")

# =====================================================
# 4. HISTORIAL DESPUÉS DEL MERGE
# =====================================================

history_after = spark.sql(f"""
DESCRIBE HISTORY {catalog}.silver.silver_incidencias
""")

print("Historial Delta:")
history_after.select(
    "version",
    "timestamp",
    "operation"
).show(truncate=False)

# =====================================================
# 5. VERSIÓN ACTUAL
# =====================================================

current_version = (
    history_after
    .select("version")
    .first()[0]
)

print(f"Versión actual: {current_version}")

# =====================================================
# 6. TIME TRAVEL - VERSIÓN ANTERIOR
# =====================================================

print(f"\n===== VERSION ANTERIOR ({version_before}) =====")

df_old = (
    spark.read
    .option("versionAsOf", version_before)
    .table(f"{catalog}.silver.silver_incidencias")
)

print(f"Registros versión anterior: {df_old.count()}")

df_old.show(10, False)

# =====================================================
# 7. TABLA ACTUAL
# =====================================================

print("\n===== VERSION ACTUAL =====")

df_current = spark.table(
    f"{catalog}.silver.silver_incidencias"
)

print(f"Registros versión actual: {df_current.count()}")

df_current.show(10, False)

# =====================================================
# 8. VALIDACIÓN FINAL
# =====================================================

print("\n===== RESUMEN =====")

print(f"Versión antes del MERGE: {version_before}")
print(f"Versión después del MERGE: {current_version}")

print(
    f"Incremento de versiones: "
    f"{current_version - version_before}"
)

Versión antes del MERGE: 3
✓ MERGE ejecutado correctamente
Historial Delta:
+-------+-------------------+---------------------------------+
|version|timestamp          |operation                        |
+-------+-------------------+---------------------------------+
|4      |2026-06-23 03:25:09|MERGE                            |
|3      |2026-06-23 03:24:55|CREATE OR REPLACE TABLE AS SELECT|
|2      |2026-06-23 03:14:31|OPTIMIZE                         |
|1      |2026-06-23 03:14:29|MERGE                            |
|0      |2026-06-23 03:08:09|CREATE OR REPLACE TABLE AS SELECT|
+-------+-------------------+---------------------------------+

Versión actual: 5

===== VERSION ANTERIOR (3) =====
Registros versión anterior: 120
+-------------+---------+-------------------+----------------+-----------------------+-----------------+--------------------------+
|id_incidencia|id_envio |tipo_incidencia    |fecha_incidencia|resolucion             |estado_resolucion|ingestion_timestamp       |

In [0]:
from pyspark.sql import functions as F

catalog = "cat_asolisr_20260622"

# =====================================================
# CREAR ESQUEMA GOLD
# =====================================================

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")

# =====================================================
# LEER TABLAS SILVER
# =====================================================

silver_envios = spark.table(
    f"{catalog}.silver.silver_envios"
)

silver_transportistas = spark.table(
    f"{catalog}.silver.silver_transportistas"
)

silver_rutas = spark.table(
    f"{catalog}.silver.silver_rutas"
)

# =====================================================
# GOLD 1 - ENVIOS POR ZONA
# =====================================================

gold_envios_zona = (
    silver_envios.alias("e")
    .join(
        silver_transportistas.alias("t"),
        on="id_transportista",
        how="left"
    )

    .withColumn("anio", F.year("fecha_envio"))
    .withColumn("mes", F.month("fecha_envio"))

    .groupBy(
        "zona",
        "anio",
        "mes"
    )

    .agg(
        F.count("id_envio").alias("total_envios"),
        F.sum("monto_flete").alias("monto_total"),
        F.sum("peso_kg").alias("peso_total")
    )

    .withColumn(
        "ticket_promedio",
        F.col("monto_total") / F.col("total_envios")
    )
)

(
    gold_envios_zona.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalog}.gold.gold_envios_zona"
    )
)

print("✓ gold_envios_zona creada")


# =====================================================
# GOLD 2 - ENVIOS POR TIPO DE RUTA
# =====================================================

gold_envios_tipo_ruta = (
    silver_envios.alias("e")
    .join(
        silver_rutas.alias("r"),
        on="id_ruta",
        how="left"
    )

    .withColumn("anio", F.year("fecha_envio"))
    .withColumn("mes", F.month("fecha_envio"))

    .groupBy(
        "tipo_ruta",
        "anio",
        "mes"
    )

    .agg(
        F.count("id_envio").alias("total_envios"),
        F.sum("monto_flete").alias("monto_total"),
        (
            F.sum("monto_flete")
            / F.count("id_envio")
        ).alias("ticket_promedio"),
        F.avg("distancia_km").alias("distancia_promedio")
    )
)

(
    gold_envios_tipo_ruta.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalog}.gold.gold_envios_tipo_ruta"
    )
)

print("✓ gold_envios_tipo_ruta creada")

# =====================================================
# VALIDACION REQUERIDA POR LA CONSIGNA
# =====================================================

total_silver_envios = silver_envios.count()

total_gold_envios = (
    spark.table(
        f"{catalog}.gold.gold_envios_zona"
    )
    .agg(
        F.sum("total_envios").alias("total")
    )
    .collect()[0]["total"]
)

diferencia = abs(
    total_silver_envios - total_gold_envios
)

print("\n========== VALIDACION ==========")
print(f"Total filas silver_envios : {total_silver_envios}")
print(f"Total envios Gold Zona    : {total_gold_envios}")
print(f"Diferencia                : {diferencia}")

if diferencia <= 10:
    print(
        "✓ VALIDACION EXITOSA (tolerancia <= 10 filas)"
    )
else:
    print(
        "✗ VALIDACION FALLIDA"
    )

# =====================================================
# CONSULTA RAPIDA
# =====================================================

print("\n===== GOLD ENVÍOS ZONA =====")
spark.table(
    f"{catalog}.gold.gold_envios_zona"
).show(20, False)

print("\n===== GOLD ENVÍOS TIPO RUTA =====")
spark.table(
    f"{catalog}.gold.gold_envios_tipo_ruta"
).show(20, False)

✓ gold_envios_zona creada
✓ gold_envios_tipo_ruta creada

========== VALIDACION ==========
Total filas silver_envios : 600
Total envios Gold Zona    : 600
Diferencia                : 0
✓ VALIDACION EXITOSA (tolerancia <= 10 filas)

===== GOLD ENVÍOS ZONA =====
+------+----+---+------------+------------------+------------------+------------------+
|zona  |anio|mes|total_envios|monto_total       |peso_total        |ticket_promedio   |
+------+----+---+------------+------------------+------------------+------------------+
|Este  |2023|12 |3           |10165.06          |1645.61           |3388.353333333333 |
|Centro|2023|9  |2           |4322.47           |749.63            |2161.235          |
|Este  |2024|4  |5           |18698.86          |1791.41           |3739.772          |
|Centro|2024|7  |4           |18938.64          |1266.82           |4734.66           |
|Norte |2024|1  |5           |24783.329999999998|1597.3300000000002|4956.665999999999 |
|Centro|2022|2  |1           |1793.

In [0]:
# =====================================================
# LIMPIEZA DEL LABORATORIO
# (DEJAR COMENTADO POR DEFECTO)
# =====================================================

# catalog = "cat_asolisr_20260622"

# -----------------------------
# TABLAS GOLD
# -----------------------------
# spark.sql(f"DROP TABLE IF EXISTS {catalog}.gold.gold_envios_zona")
# spark.sql(f"DROP TABLE IF EXISTS {catalog}.gold.gold_envios_tipo_ruta")

# -----------------------------
# TABLAS SILVER
# -----------------------------
# spark.sql(f"DROP TABLE IF EXISTS {catalog}.silver.silver_envios")
# spark.sql(f"DROP TABLE IF EXISTS {catalog}.silver.silver_clientes_logistica")
# spark.sql(f"DROP TABLE IF EXISTS {catalog}.silver.silver_incidencias")
# spark.sql(f"DROP TABLE IF EXISTS {catalog}.silver.silver_rutas")
# spark.sql(f"DROP TABLE IF EXISTS {catalog}.silver.silver_transportistas")

# -----------------------------
# TABLAS BRONZE
# -----------------------------
# spark.sql(f"DROP TABLE IF EXISTS {catalog}.bronze.bronze_envios")
# spark.sql(f"DROP TABLE IF EXISTS {catalog}.bronze.bronze_clientes_logistica")
# spark.sql(f"DROP TABLE IF EXISTS {catalog}.bronze.bronze_incidencias_2023")
# spark.sql(f"DROP TABLE IF EXISTS {catalog}.bronze.bronze_incidencias_2024")
# spark.sql(f"DROP TABLE IF EXISTS {catalog}.bronze.bronze_rutas")
# spark.sql(f"DROP TABLE IF EXISTS {catalog}.bronze.bronze_transportistas")

# -----------------------------
# ESQUEMAS (opcional)
# Ejecutar solo si todas las
# tablas fueron eliminadas.
# -----------------------------
# spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.gold CASCADE")
# spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.silver CASCADE")
# spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.bronze CASCADE")

# -----------------------------
# CATÁLOGO COMPLETO (opcional)
# Ejecutar solo si deseas borrar
# todo el laboratorio.
# -----------------------------
# spark.sql(f"DROP CATALOG IF EXISTS {catalog} CASCADE")

# print("Limpieza completada")

In [0]:
storage_account = "asolisrstad3"

landing_path = f"abfss://landing@{storage_account}.dfs.core.windows.net"

display(dbutils.fs.ls(landing_path))

---------------------------------------------------------------------------
ExecutionError                            Traceback (most recent call last)
File <command-5553695758101288>, line 5
      1 storage_account = "asolisrstad3"
      3 landing_path = f"abfss://landing@{storage_account}.dfs.core.windows.net"
----> 5 display(dbutils.fs.ls(landing_path))

File /databricks/python_shell/lib/dbruntime/remotefshandler/RemoteFsHandler.py:56, in prettify_exception_message.<locals>.f_with_exception_handling(*args, **kwargs)
     52     pass
     54 error_exception = ExecutionError(str(e))
---> 56 raise patch_exception_with_error_details(
     57     error_exception,
     58     DriverErrorCode.REMOTE_FS_HANDLER_EXECUTION_ERROR  # type: ignore[attr-defined]
     59 ) from None

ExecutionError: (shaded.databricks.azurebfs.org.apache.hadoop.fs.azurebfs.contracts.exceptions.KeyProviderException) Failure to initialize configuration for storage account asolisrstad3.dfs.core.windows.net: Invalid c

In [0]:
display(dbutils.fs.ls(f"/Volumes/{catalog}/default/vol_landing"))

path,name,size,modificationTime
dbfs:/Volumes/cat_asolisr_20260622/default/vol_landing/clientes_logistica.csv,clientes_logistica.csv,3800,1782181977000
dbfs:/Volumes/cat_asolisr_20260622/default/vol_landing/envios.csv,envios.csv,56032,1782181977000
dbfs:/Volumes/cat_asolisr_20260622/default/vol_landing/incidencias_2023.csv,incidencias_2023.csv,9740,1782181977000
dbfs:/Volumes/cat_asolisr_20260622/default/vol_landing/incidencias_2024.csv,incidencias_2024.csv,6415,1782181977000
dbfs:/Volumes/cat_asolisr_20260622/default/vol_landing/rutas.csv,rutas.csv,1312,1782181977000
dbfs:/Volumes/cat_asolisr_20260622/default/vol_landing/transportistas.csv,transportistas.csv,1232,1782181977000
